In [1]:
from langchain_community.document_loaders import  TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models.base import init_chat_model
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()


d:\RAG_Learning_Repo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
loader=TextLoader("langchain_crewai_dataset.txt")
documents=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)
chunks=text_splitter.split_documents(documents)
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embeddings)
retriever=vectorstore.as_retriever()
llm=init_chat_model("groq:openai/gpt-oss-safeguard-20b")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1390.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
decomposition_prompt = PromptTemplate.from_template("""You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions from this question: {question}""")
decomposite_chain=decomposition_prompt |llm| StrOutputParser()
query="What are the key features of LangChain and how does it compare to other similar frameworks in terms of ease of use and functionality?"
response=sub_questions=decomposite_chain.invoke({"question": query})
print(response)




Here’s a way to split the original question into more focused sub‑questions:

1. **What are the core features and building blocks of LangChain?**  
   • Chain construction, memory management, prompt templates, etc.

2. **Which other LLM‑oriented frameworks are commonly compared to LangChain (e.g., LlamaIndex, PromptLayer, Haystack, etc.) and what are their main capabilities?**

3. **In terms of developer experience and ease of use, how does LangChain compare to those frameworks?**  
   • Installation, documentation, learning curve, community support.

4. **In terms of functional richness (e.g., extensibility, integration options, performance, customization), how does LangChain stack up against the alternatives?**


In [7]:
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposite_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = decomposite_chain.invoke({"question": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

In [8]:

query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: **Decomposed Sub‑Questions**
A: I’m happy to help break the question down, but I need the complex question you want decomposed. Could you please provide it? Once I have that, I can split it into 2–4 smaller, focused sub‑questions.

Q: **Memory Architecture**
A: **Decomposed sub‑questions for “Memory Architecture”**

1. **What is memory architecture and why is it important in a computer system?**  
   - Define the term and explain its role in overall system design, performance, and reliability.

2. **What are the primary layers of a typical memory hierarchy (registers, cache, main memory, secondary storage) and how do they differ in speed, capacity, and cost?**  
   - Describe the purpose of each layer and how data moves between them.

3. **How do cache characteristics (size, associativity, line size, latency) influence processor performance, and what strategies are used to manage cache coherence and replacement?**  
   - Include examples of L1, L2, and L3 cache beha